# Climate Analysis

**Purpose:** Analyze climate variation impact on heating demand (2010-2019), compare actual vs calculated consumption.

**Project imports:** `project.model`, `project.utils`.

**Inputs:** Building stock, climate data.

**Outputs:** Climate-adjusted consumption comparison figures.

In [ ]:
from pandas import Series, concat

from project.model import get_inputs
from project.utils import make_plot
from project.input.resources import resources_data

In [ ]:
output = get_inputs(variables=['buildings', 'energy_prices'], building_stock='project/input/stock/buildingstock_sdes2018_medium_3.csv')

In [ ]:
buildings, prices = output['buildings'], output['energy_prices']
price = prices.loc[2018, :]

In [ ]:
# calibration
buildings.calculate_consumption(price)
buildings.coefficient_consumption

## Outdoor temperature

In [ ]:
result = {}
for year in range(2010, 2020):
    buildings.calculate_consumption(price, climate=year)
    result.update({year: buildings.heat_consumption_calib.sum() / 10**9})

result = Series(result).rename('Calculated')

In [ ]:
result = concat((result, resources_data['consumption_total_hist_climate']), axis=1)

In [ ]:
make_plot(result, 'Consumption (TWh)')

Conclusion: the energy model reproduces well the impact of climate on aggregated energy consumption.
As the energy performance of the building stock is fixed in these calculation, it is normal to underestimate the consumption in the past.

## Indoor temperature

In [ ]:
result = {}
for temp_indoor in range(17, 25):
    buildings.calculate_consumption(price, temp_indoor=temp_indoor)
    result.update({temp_indoor: buildings.heat_consumption_calib.sum() / 10**9})

result = Series(result).rename('Calculated')
result

In [ ]:
result.diff() / result